# Online Game Price Prediction Using Machine Learning

**Course:** Machine Learning (PBL)  
**Team:** Saish Nehe (2124UMLM1051), Nikhil Jadhav (2124UMLM1096), Omprakash Wagh (2124UMLM1073), Tushar Borawake (2124UMLM1037)

## Aim
Estimate an observed Steam game price (`price`, EUR) from genre, launch year, review count, and general review summary. The source records were collected in March 2025, so this notebook does **not** predict live or future prices.

**Dataset source:** [Steam Games - UOC Project, Zenodo](https://zenodo.org/records/15121027). Run `python scripts/download_data.py` from the repository root before running this notebook.

**Leakage control:** `discount` and `original_price` are deliberately not used as input features because they directly relate to the target price.

In [ ]:
from pathlib import Path
import sys
ROOT = Path.cwd()
if not (ROOT / "src").exists():
    ROOT = ROOT.parent
sys.path.insert(0, str(ROOT / "src"))

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from data_utils import read_dataset, clean_dataset, model_features, FEATURE_COLUMNS, TARGET_COLUMN
from modeling import train_and_evaluate

pd.set_option("display.max_columns", 50)
sns.set_theme(style="whitegrid")
DATA_PATH = ROOT / "data" / "raw" / "steam-games.csv"
print("Project root:", ROOT)
print("Dataset path:", DATA_PATH)

## 1. Load and understand the dataset

In [ ]:
raw = read_dataset(DATA_PATH)
print("Raw shape:", raw.shape)
display(raw.head())
print("\nColumns:")
print(raw.columns.tolist())
print("\nData types:")
display(raw.dtypes.to_frame("dtype"))

## 2. Data quality inspection

In [ ]:
quality = pd.DataFrame({
    "missing_count": raw.isna().sum(),
    "missing_percent": (raw.isna().mean() * 100).round(2),
    "unique_values": raw.nunique(dropna=True),
})
display(quality.sort_values("missing_count", ascending=False))
print("Exact duplicate rows:", int(raw.duplicated().sum()))
if "is_error" in raw.columns:
    print("Scraping error value counts:")
    display(raw["is_error"].value_counts(dropna=False))

## 3. Clean the data and inspect target

In [ ]:
cleaned = clean_dataset(raw)
X, y = model_features(cleaned)
print("Cleaned shape:", cleaned.shape)
print("Model features:", FEATURE_COLUMNS)
print("Target:", TARGET_COLUMN)
display(cleaned[FEATURE_COLUMNS + [TARGET_COLUMN]].head())
display(y.describe().to_frame("price_eur"))

## 4. Exploratory Data Analysis

Interpret each plot in the report using the actual chart. Do not assume trends before observing the results.

In [ ]:
fig, ax = plt.subplots(figsize=(8, 4))
sns.histplot(cleaned[TARGET_COLUMN], bins=40, kde=False, color="steelblue", ax=ax)
ax.set_title("Distribution of observed game prices")
ax.set_xlabel("Price (EUR)")
plt.tight_layout()
plt.show()

In [ ]:
genre_summary = (cleaned.groupby("genres", dropna=False)
                 .agg(game_count=(TARGET_COLUMN, "size"), median_price_eur=(TARGET_COLUMN, "median"))
                 .sort_values("game_count", ascending=False))
display(genre_summary.head(15))
plot_genres = genre_summary.head(12).sort_values("median_price_eur")
fig, ax = plt.subplots(figsize=(9, 5))
plot_genres["median_price_eur"].plot(kind="barh", ax=ax)
ax.set_title("Median observed price for the 12 most frequent genre strings")
ax.set_xlabel("Median price (EUR)")
ax.set_ylabel("Genre string")
plt.tight_layout()
plt.show()

In [ ]:
fig, ax = plt.subplots(figsize=(8, 4))
sns.scatterplot(data=cleaned.sample(min(len(cleaned), 1500), random_state=42),
                x="reviews", y=TARGET_COLUMN, alpha=0.55, ax=ax)
ax.set_title("Review count vs observed game price (sample if dataset is large)")
ax.set_xlabel("Number of reviews")
ax.set_ylabel("Price (EUR)")
plt.tight_layout()
plt.show()

## 5. Feature engineering and modelling approach

- `launch_year` is extracted from `launch_date` by the cleaning function.
- Numeric inputs: `launch_year`, `reviews`.
- Categorical inputs: `genres`, `general_reviews`, encoded with one-hot encoding.
- Missing input values are handled within a scikit-learn pipeline.
- `discount`, `original_price`, game title, URL, and long free-text description are excluded from the baseline model.
- Data is split 80/20. Models are selected by lowest mean 5-fold cross-validation MAE on training data. The held-out test set is used once for final evaluation.

Metrics: MAE (EUR), RMSE (EUR), and R². Lower MAE/RMSE are better; R² closer to 1 generally indicates more explained variance.

In [ ]:
from modeling import candidate_models
print("Models to compare:", ", ".join(candidate_models().keys()))
print("Selection rule: lowest mean 5-fold cross-validation MAE on the training split.")

## 6. Train the selected model and evaluate once on the test set

In [ ]:
# This function repeats the consistent train/evaluate workflow and saves the fitted pipeline + metrics.
result = train_and_evaluate(
    data_path=DATA_PATH,
    model_path=ROOT / "models" / "best_model.joblib",
    metadata_path=ROOT / "models" / "model_metadata.json",
    reports_path=ROOT / "reports",
)
display(result["comparison"])
print("Final held-out test metrics:")
for key, value in result["metrics"].items():
    print(f"{key}: {value}")

## 7. Actual vs predicted prices

The following chart uses the final selected model and the held-out predictions written by the training function.

In [ ]:
pred_path = ROOT / "reports" / "test_predictions.csv"
preds = pd.read_csv(pred_path)
fig, ax = plt.subplots(figsize=(6, 6))
ax.scatter(preds["actual_price_eur"], preds["predicted_price_eur"], alpha=0.6)
upper = max(preds["actual_price_eur"].max(), preds["predicted_price_eur"].max())
ax.plot([0, upper], [0, upper], linestyle="--")
ax.set_title("Actual vs predicted prices (held-out test set)")
ax.set_xlabel("Actual price (EUR)")
ax.set_ylabel("Predicted price (EUR)")
plt.tight_layout()
plt.show()

## 8. Feature importance (interpretation only)

This permutation-importance analysis estimates how much the held-out MAE changes when each original input column is shuffled. It is for interpreting the selected model, not for repeatedly tuning the model against the test set.

In [ ]:
import joblib
from sklearn.inspection import permutation_importance
from sklearn.model_selection import train_test_split

# Recreate the exact deterministic split used during training.
X_imp_train, X_imp_test, y_imp_train, y_imp_test = train_test_split(
    X, y, test_size=0.20, random_state=42
)
selected_pipeline = joblib.load(ROOT / "models" / "best_model.joblib")
perm = permutation_importance(
    selected_pipeline, X_imp_test, y_imp_test,
    scoring="neg_mean_absolute_error", n_repeats=5, random_state=42, n_jobs=1
)
importance = pd.DataFrame({
    "feature": X_imp_test.columns,
    "mae_increase_when_shuffled_eur": perm.importances_mean,
    "std_eur": perm.importances_std,
}).sort_values("mae_increase_when_shuffled_eur")
display(importance)
ax = importance.set_index("feature")["mae_increase_when_shuffled_eur"].plot(kind="barh", figsize=(8, 4))
ax.set_title("Permutation importance on held-out test split")
ax.set_xlabel("Increase in MAE when shuffled (EUR)")
ax.set_ylabel("Input feature")
plt.tight_layout()
plt.show()

## 9. Business insights and limitations

After running all cells, write evidence-based observations here:

1. Which model had the lowest 5-fold validation MAE?
2. What were the final test MAE, RMSE, and R²?
3. Which price ranges or categories seem harder to estimate? Use actual plots/errors.
4. What data quality issues were observed and how were they handled?
5. Why should the application not be treated as a current/live price quote?

**Limitations:** prices represent a March 2025 snapshot from the Spanish-language Steam store; game prices differ by country and change over time. Features are limited and performance is not a guarantee. Cite the original dataset and do not claim real-world benefits that were not measured.

## 10. Application launch

From a terminal at the repository root, run:

```bash
streamlit run app.py
```

The app requires `models/best_model.joblib` and `models/model_metadata.json`, which are created by the training workflow.